In [ ]:
import os
import warnings
import json
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,classification_report,confusion_matrix,roc_auc_score,roc_curve
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import mlflow
import mlflow.pytorch
os.makedirs('artifacts',exist_ok=True)
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('PyTorch:', torch.__version__)

In [ ]:
df=pd.read_csv('UCI_Credit_Card.csv')
print('Shape:',df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
df.describe().T

In [ ]:
target_col='default.payment.next.month'
counts=df[target_col].value_counts()
plt.figure(figsize=(5,4))
counts.plot(kind='bar',color=['steelblue','tomato'])
plt.title('Target Class Distribution')
plt.xlabel('Default(1=Yes,0=No)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('artifacts/class_distribution.png',dpi=100)
plt.show()

print(counts)

In [ ]:
plt.figure(figsize=(16,12))
corr=df.drop(columns=['ID']).corr()
sns.heatmap(corr,cmap='coolwarm',annot=False,linewidth=0.3)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('artifacts/correlation_heatmap.png',dpi=100)
plt.show()

In [ ]:
features=['LIMIT_BAL','AGE','BILL_AMT1','PAY_AMT1']
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,col in zip(axes,features):
  df[col].hist(bins=30,ax=ax,color='steelblue',edgecolor='white')
  ax.set_title(col)
plt.suptitle('Key Feature Distributions')
plt.tight_layout()
plt.savefig('artifacts/feature_distriution.png',dpi=100)
plt.show

In [ ]:
plt.figure(figsize=(15, 10))

sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='Pastel2', linewidths=2)

plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df=df.drop(columns=['ID'])

df['EDUCATION']=df['EDUCATION'].replace({0:4,5:4,6:4})
df['MARRIAGE']=df['MARRIAGE'].replace({0:3})




In [ ]:
# Total times payment was delayed
df['TOTAL_DELAY'] = (df[['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']] > 0).sum(axis=1)

# Maximum delay across all months
df['MAX_DELAY'] = df[['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']].max(axis=1)

# Is currently delayed (most recent month)
df['CURRENTLY_DELAYED'] = (df['PAY_0'] > 0).astype(int)

# Consecutive delays — how many months in a row delayed
df['CONSECUTIVE_DELAY'] = (
    (df['PAY_0'] > 0).astype(int) +
    (df['PAY_2'] > 0).astype(int) +
    (df['PAY_3'] > 0).astype(int)
)

In [ ]:

# Average bill amount across 6 months
df['AVG_BILL'] = df[['BILL_AMT1','BILL_AMT2','BILL_AMT3',
                      'BILL_AMT4','BILL_AMT5','BILL_AMT6']].mean(axis=1)

# Credit utilization — how much of limit is used
df['UTILIZATION'] = df['AVG_BILL'] / (df['LIMIT_BAL'] + 1)

# Bill trend — is bill increasing or decreasing
df['BILL_TREND'] = df['BILL_AMT1'] - df['BILL_AMT6']

# Max bill amount
df['MAX_BILL'] = df[['BILL_AMT1','BILL_AMT2','BILL_AMT3',
                      'BILL_AMT4','BILL_AMT5','BILL_AMT6']].max(axis=1)

# Total amount paid across 6 months
df['TOTAL_PAID'] = df[['PAY_AMT1','PAY_AMT2','PAY_AMT3',
                        'PAY_AMT4','PAY_AMT5','PAY_AMT6']].sum(axis=1)

# Average payment
df['AVG_PAID'] = df['TOTAL_PAID'] / 6


In [ ]:


# Recent payment trend — paying more or less recently
df['PAY_TREND'] = df['PAY_AMT1'] - df['PAY_AMT6']

# Months with zero payment
df['ZERO_PAY_COUNT'] = (
    df[['PAY_AMT1','PAY_AMT2','PAY_AMT3',
        'PAY_AMT4','PAY_AMT5','PAY_AMT6']] == 0
).sum(axis=1)

# Remaining balance after payment each month
df['BAL_AFTER_PAY1'] = df['BILL_AMT1'] - df['PAY_AMT1']
df['BAL_AFTER_PAY2'] = df['BILL_AMT2'] - df['PAY_AMT2']
df['BAL_AFTER_PAY3'] = df['BILL_AMT3'] - df['PAY_AMT3']


In [ ]:
# Average remaining balance
df['AVG_REMAINING_BAL'] = (
    df['BAL_AFTER_PAY1'] +
    df['BAL_AFTER_PAY2'] +
    df['BAL_AFTER_PAY3']
) / 3

# Is balance growing
df['BAL_GROWING'] = (df['BAL_AFTER_PAY1'] > df['BAL_AFTER_PAY3']).astype(int)

# Age groups
df['AGE_GROUP'] = pd.cut(
    df['AGE'],
    bins=[0, 25, 35, 45, 60, 100],
    labels=[1, 2, 3, 4, 5]
).astype(int)

# High education flag
df['HIGH_EDUCATION'] = (df['EDUCATION'] <= 2).astype(int)

# Limit per age — credit relative to age
df['LIMIT_PER_AGE'] = df['LIMIT_BAL'] / df['AGE']

In [ ]:

X=df.drop(columns=[target_col]).values.astype(np.float32)
y=df[target_col].values.astype(np.float32)

print('X Shape:',X.shape)
print('Y Shape:',y.shape) 

In [ ]:
X_temp,X_test,y_temp,y_test=train_test_split(
    X, y ,test_size=0.15,random_state=42,stratify=y
)
X_train,X_val,y_train,y_val=train_test_split(
    X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp
)
print('Train:',X_train.shape)
print('Test:',X_test.shape)
print('Val:',X_val.shape)

In [ ]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
X_val=scaler.transform(X_val)

with open('artifacts/scaler.pkl','wb') as f:
  pickle.dump(scaler,f)
print('scaler saved')

In [ ]:
def to_tensor(X,y):
  return (
      torch.tensor(X,dtype=torch.float32),
      torch.tensor(y,dtype=torch.float32).unsqueeze(1)
  )

X_train_t,y_train_t=to_tensor(X_train,y_train)
X_val_t,y_val_t=to_tensor(X_val,y_val)
X_test_t,y_test_t=to_tensor(X_test,y_test)

BATCH_SIZE=128
train_loader=DataLoader(TensorDataset(X_train_t,y_train_t),batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(TensorDataset(X_val_t,y_val_t),batch_size=BATCH_SIZE)
test_loader=DataLoader(TensorDataset(X_test_t,y_test_t),batch_size=BATCH_SIZE)

print('DataLoaders ready.')

In [ ]:
num_no_default = (y_train == 0).sum()
num_default    = (y_train == 1).sum()
pos_weight     = torch.tensor([num_no_default / num_default],
                               dtype=torch.float32).to(device)
criterion      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f'No Default count : {num_no_default}')
print(f'Default count    : {num_default}')
print(f'Class weight     : {pos_weight.item():.4f}')

In [ ]:
INPUT_DIM=X_train.shape[1]


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1     = nn.Linear(INPUT_DIM, 128)
        self.fc2     = nn.Linear(128, 64)
        self.fc3     = nn.Linear(64, 32)       
        self.fc4     = nn.Linear(32, 1)        
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.bn1     = nn.BatchNorm1d(128)
        self.bn2     = nn.BatchNorm1d(64)
        self.bn3     = nn.BatchNorm1d(32)      

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.relu(self.bn3(self.fc3(x)))  
        x = self.dropout(x)
        x = self.fc4(x)                        
        return x

def apply_weight_init(model,init_type='he'):
  for layer in model.modules():
    if isinstance(layer,nn.Linear):
      if init_type=='he':
        nn.init.kaiming_uniform_(layer.weight,nonlinearity='relu')
      elif init_type=='xavier':
        nn.init.xavier_uniform_(layer.weight)
      nn.init.zeros_(layer.bias)
  return model

model=MLP()
model=apply_weight_init(model,'he')

dummy_input=torch.randn(4,INPUT_DIM)
output = model(dummy_input)
print('Model output shape:',output.shape)
print(model)

In [ ]:
def train_one_epoch(model,loader,optimizer,criterion):
  model.train()
  total_loss=0
  total_correct=0
  total_samples=0

  for X_batch,y_batch in loader:
    X_batch=X_batch.to(device)
    y_batch=y_batch.to(device)

    #forward pass
    output=model(X_batch)
    loss=criterion(output,y_batch)

    #backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    predictions=(torch.sigmoid(output)>0.5).float()
    total_loss += loss.item()
    total_correct += (predictions == y_batch).sum().item()
    total_samples +=len(y_batch)

  avg_loss=total_loss / len(loader)
  accuracy=total_correct / total_samples
  return avg_loss , accuracy

In [ ]:
def evaluate(model,loader,criterion):
  model.eval()
  total_loss=0
  total_correct=0
  total_samples=0
  all_probs=[]
  all_labels=[]

  with torch.no_grad():
    for X_batch,y_batch in loader:
      X_batch=X_batch.to(device)
      y_batch=y_batch.to(device)

      output =model(X_batch)
      loss=criterion(output,y_batch)
      probs=torch.sigmoid(output)
      predictions=(probs>0.5).float()

      total_loss += loss.item()
      total_correct += (predictions == y_batch).sum().item()
      total_samples += len(y_batch)
      all_probs.extend(probs.cpu().numpy().flatten())
      all_labels.extend(y_batch.cpu().numpy().flatten())
  avg_loss=total_loss / len(loader)
  accuracy=total_correct/total_samples
  return avg_loss,accuracy,np.array(all_probs),np.array(all_labels)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience     = patience
        self.min_delta    = min_delta   # ← minimum improvement to count
        self.best_loss    = np.inf
        self.counter      = 0
        self.best_weights = None

    def check(self, val_loss, model):
        if np.isnan(val_loss):
            self.counter += 1
            return self.counter >= self.patience

        # Only count as improvement if drop is MORE than min_delta
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1

        if self.counter >= self.patience:
            return True
        return False

In [ ]:
def train_model(model, train_loader, val_loader, lr=1e-3,
                weight_decay=1e-4, n_epochs=80, patience=10):
    model     = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    stopper = EarlyStopping(patience=7, min_delta=0.001)

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for epoch in range(n_epochs):
        tr_loss, tr_acc       = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        train_accs.append(tr_acc)
        val_accs.append(vl_acc)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{n_epochs} | '
                  f'Train Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} | '
                  f'Val Acc: {vl_acc:.4f}')

        if stopper.check(vl_loss, model):
            print(f'Early Stopping at Epoch {epoch+1}')
            break

    model.load_state_dict(stopper.best_weights)

    history = {
        'train_loss': train_losses,
        'val_loss':   val_losses,
        'train_acc':  train_accs,
        'val_acc':    val_accs
    }
    return model, history

print('Training utilities ready.')

In [ ]:
model=MLP()
model=apply_weight_init(model,'he')
model,history=train_model(model,train_loader,val_loader,n_epochs=20,patience=10)
print('test run complete ')

In [ ]:
def plot_history(history):
  fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))

  ax1.plot(history['train_loss'],label='Train Loss',color='steelblue')
  ax1.plot(history['val_loss'], label='Val Loss', color='tomato')
  ax1.set_title('Loss vs Epochs')
  ax1.set_xlabel('Epoch')
  ax1.set_ylabel('Loss')
  ax1.legend()

  ax2.plot(history['train_acc'],label='Train Avvuracy',color='steelblue')
  ax2.plot(history['val_acc'],label='Val A ccuracy',color='tomato')
  ax2.set_title('Accuracy vs Epochs')
  ax2.set_xlabel('Epoch')
  ax2.set_ylabel('Accuracy')
  ax2.legend()

  plt.tight_layout()
  plt.savefig('artifacts/training_history.png',dpi=100)
  plt.show()

plot_history(history)

In [ ]:
mlflow.set_experiment('credit_default_experiments')

EXPERIMENTS = [
    ('he_adam',         'he',      'adam',   1e-3, 1e-4),
    ('xavier_adam',     'xavier',  'adam',   1e-3, 1e-4),
    ('he_adamw',        'he',      'adamw',  1e-3, 1e-4),
    ('he_adam_lr_low',  'he',      'adam',   1e-4, 1e-4),
    ('he_adam_wd_high', 'he',      'adam',   1e-3, 1e-3),
    ('xavier_adamw',    'xavier',  'adamw',  1e-3, 1e-4),
]

results = []

for run_name, init_type, opt_name, lr, wd in EXPERIMENTS:
    print(f'\n--- Run: {run_name} ---')

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'weight_init':  init_type,
            'optimizer':    opt_name,
            'lr':           lr,
            'weight_decay': wd,
            'batch_size':   BATCH_SIZE,
            'hidden_units': '128-64-32',
            'dropout':      0.3
        })

        model = MLP()
        model = apply_weight_init(model, init_type)
        model = model.to(device)

        # Fix — correct variable name, no typo
        if opt_name == 'adamw':
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        else:
            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

        stopper = EarlyStopping(patience=7, min_delta=0.001)

        for epoch in range(80):
            tr_loss, tr_acc       = train_one_epoch(model, train_loader, optimizer, criterion)
            vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)

            mlflow.log_metrics({
                'train_loss': tr_loss,
                'val_loss':   vl_loss,
                'train_acc':  tr_acc,
                'val_acc':    vl_acc
            }, step=epoch)

            if stopper.check(vl_loss, model):
                print(f'  Early stopping at epoch {epoch+1}')
                break

        model.load_state_dict(stopper.best_weights)

        _, val_acc, val_probs, val_labels = evaluate(model, val_loader, criterion)
        val_auc = roc_auc_score(val_labels, val_probs)

        mlflow.log_metrics({'final_val_acc': val_acc, 'final_val_auc': val_auc})
        mlflow.pytorch.log_model(model, 'model')

        results.append({
            'run':     run_name,
            'val_acc': round(val_acc, 4),
            'val_auc': round(val_auc, 4)
        })
        print(f'  Val Acc: {val_acc:.4f}  |  Val AUC: {val_auc:.4f}')

results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
print('\n=== Experiment Comparison ===')
print(results_df.to_string(index=False))

In [ ]:
from sklearn.metrics import precision_score, recall_score

mlflow.set_experiment('credit_default_random_search')

SEARCH_SPACE = {
    'lr':           [1e-3, 5e-4, 1e-4],
    'weight_decay': [0.0,  1e-4, 1e-3],
    'batch_size':   [32,64,128],
    'dropout':      [0.2,  0.3,  0.4],
}

N_TRIALS     = 10
best_recall  = 0
best_model   = None
best_config  = None
trial_results = []

random.seed(42)


print(f'Class weight for default: {pos_weight.item():.4f}')

for trial in range(N_TRIALS):
    cfg      = {k: random.choice(v) for k, v in SEARCH_SPACE.items()}
    run_name = f'trial_{trial+1}'
    print(f'\nTrial {trial+1}/{N_TRIALS} | Config: {cfg}')

    tl = DataLoader(TensorDataset(X_train_t, y_train_t),
                    batch_size=cfg['batch_size'], shuffle=True)
    vl = DataLoader(TensorDataset(X_val_t, y_val_t),
                    batch_size=cfg['batch_size'])

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(cfg)

        model     = MLP()
        model     = apply_weight_init(model, 'he')
        model     = model.to(device)
        optimizer = optim.AdamW(model.parameters(),
                                lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        stopper = EarlyStopping(patience=7, min_delta=0.001)

        for epoch in range(60):
            tr_loss, tr_acc       = train_one_epoch(model, tl, optimizer, criterion)
            vl_loss, vl_acc, _, _ = evaluate(model, vl, criterion)

            mlflow.log_metrics({
                'train_loss': tr_loss,
                'val_loss':   vl_loss,
                'train_acc':  tr_acc,
                'val_acc':    vl_acc
            }, step=epoch)

            if stopper.check(vl_loss, model):
                print(f'  Early stopping at epoch {epoch+1}')
                break

        model.load_state_dict(stopper.best_weights)

        _, val_acc, val_probs, val_labels = evaluate(model, vl, criterion)
        val_preds     = (val_probs > 0.5).astype(int)
        val_auc       = roc_auc_score(val_labels,  val_probs)
        val_precision = precision_score(val_labels, val_preds, zero_division=0)
        val_recall    = recall_score(val_labels,    val_preds, zero_division=0)

        mlflow.log_metrics({
            'final_val_acc':       round(val_acc,       4),
            'final_val_auc':       round(val_auc,       4),
            'final_val_precision': round(val_precision, 4),
            'final_val_recall':    round(val_recall,    4),
        })

        trial_results.append({
            'trial':     trial + 1,
            'config':    cfg,
            'val_acc':   round(val_acc,       4),
            'val_auc':   round(val_auc,       4),
            'precision': round(val_precision, 4),
            'recall':    round(val_recall,    4),
        })

        print(f'  Val Acc  : {val_acc:.4f}')
        print(f'  Val AUC  : {val_auc:.4f}')
        print(f'  Precision: {val_precision:.4f}')
        print(f'  Recall   : {val_recall:.4f}')

        if val_recall > best_recall:
            best_recall = val_recall
            best_config = cfg
            best_model  = model
            print(f'   New best model — Recall: {best_recall:.4f}')

print('\nTrial Comparison ')
results_df = pd.DataFrame([{
    'trial':     r['trial'],
    'lr':        r['config']['lr'],
    'wd':        r['config']['weight_decay'],
    'batch':     r['config']['batch_size'],
    'dropout':   r['config']['dropout'],
    'acc':       r['val_acc'],
    'auc':       r['val_auc'],
    'precision': r['precision'],
    'recall':    r['recall'],
} for r in trial_results]).sort_values('recall', ascending=False)

print(results_df.to_string(index=False))
print(f'\nBest Config : {best_config}')
print(f'Best Recall : {best_recall:.4f}')

Best Model Metrics

In [ ]:

print(f"Retraining best config: {best_config}")

tl_best = DataLoader(TensorDataset(X_train_t, y_train_t),
                     batch_size=best_config['batch_size'], shuffle=True)
vl_best = DataLoader(TensorDataset(X_val_t, y_val_t),
                     batch_size=best_config['batch_size'])

best_model_retrain = MLP()
best_model_retrain = apply_weight_init(best_model_retrain, 'he')
best_model_retrain = best_model_retrain.to(device)

optimizer_best = optim.AdamW(best_model_retrain.parameters(),
                              lr=best_config['lr'],
                              weight_decay=best_config['weight_decay'])
stopper_best = EarlyStopping(patience=7, min_delta=0.001)

history_best = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  []
}

for epoch in range(60):
    tr_loss, tr_acc       = train_one_epoch(best_model_retrain, tl_best, optimizer_best, criterion)
    vl_loss, vl_acc, _, _ = evaluate(best_model_retrain, vl_best, criterion)

    history_best['train_loss'].append(tr_loss)
    history_best['val_loss'].append(vl_loss)
    history_best['train_acc'].append(tr_acc)
    history_best['val_acc'].append(vl_acc)

    if stopper_best.check(vl_loss, best_model_retrain):
        print(f'Early stopping at epoch {epoch+1}')
        break

best_model_retrain.load_state_dict(stopper_best.best_weights)
print("Retrain complete.")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# ── Get test set predictions from retrained best model ──
_, test_acc, test_probs, test_labels = evaluate(best_model_retrain, test_loader, criterion)
test_auc = roc_auc_score(test_labels, test_probs)
fpr, tpr, _ = roc_curve(test_labels, test_probs)

epochs = range(1, len(history_best['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Loss curve ──
axes[0].plot(epochs, history_best['train_loss'], color='steelblue', label='Train Loss')
axes[0].plot(epochs, history_best['val_loss'],   color='tomato',    label='Val Loss')
axes[0].set_title(f'Loss Curve — Best Config\nlr={best_config["lr"]} | wd={best_config["weight_decay"]} | batch={best_config["batch_size"]} | dropout={best_config["dropout"]}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Plot 2: Accuracy curve ──
axes[1].plot(epochs, history_best['train_acc'], color='steelblue', label='Train Accuracy')
axes[1].plot(epochs, history_best['val_acc'],   color='tomato',    label='Val Accuracy')
axes[1].set_title('Accuracy Curve — Best Config')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── Plot 3: ROC-AUC curve ──
axes[2].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {test_auc:.4f})')
axes[2].plot([0, 1], [0, 1], '--', color='gray', label='Random Classifier')
axes[2].fill_between(fpr, tpr, alpha=0.08, color='steelblue')
axes[2].set_title('ROC-AUC Curve — Best Model (Test Set)')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle(f'Best Model Performance  |  Test AUC: {test_auc:.4f}  |  Test Acc: {test_acc:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/best_model_curves.png', dpi=100)
plt.show()

print(f'\nTest Accuracy : {test_acc:.4f}')
print(f'Test ROC-AUC  : {test_auc:.4f}')

In [ ]:
_, test_acc, test_probs, test_labels = evaluate(best_model, test_loader, criterion)
test_auc       = roc_auc_score(test_labels,  test_probs)
test_preds     = (test_probs > 0.5).astype(int)
test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall    = recall_score(test_labels,    test_preds, zero_division=0)

print(' Test Set Results ')
print(f'Accuracy  : {test_acc:.4f}')
print(f'ROC-AUC   : {test_auc:.4f}')
print(f'Precision : {test_precision:.4f}')
print(f'Recall    : {test_recall:.4f}')
print()
print(classification_report(test_labels, test_preds,
      target_names=['No Default', 'Default']))

In [ ]:
print('Threshold Analysis \n')

thresholds = [0.5, 0.45, 0.40, 0.35, 0.30, 0.25]

for thresh in thresholds:
    preds     = (test_probs > thresh).astype(int)
    precision = precision_score(test_labels, preds, zero_division=0)
    recall    = recall_score(test_labels,    preds, zero_division=0)
    acc       = accuracy_score(test_labels,  preds)

    print(f'Threshold: {thresh} | Acc: {acc:.4f} | '
          f'Precision: {precision:.4f} | Recall: {recall:.4f}')

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix — Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('artifacts/confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(test_labels, test_probs)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color='steelblue', label=f'AUC = {test_auc:.4f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.title('ROC Curve — Test Set')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/roc_curve.png', dpi=100)
plt.show()

In [ ]:
errors_idx  = np.where(test_preds != test_labels.astype(int))[0]
correct_idx = np.where(test_preds == test_labels.astype(int))[0]

plt.figure(figsize=(8, 4))
plt.hist(test_probs[correct_idx], bins=30, alpha=0.6, label='Correct', color='steelblue')
plt.hist(test_probs[errors_idx],  bins=30, alpha=0.6, label='Wrong',   color='tomato')
plt.title('Confidence: Correct vs Wrong Predictions')
plt.xlabel('Predicted Probability of Default')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/error_analysis.png', dpi=100)
plt.show()

print(f'Total samples    : {len(test_labels)}')
print(f'Correct          : {len(correct_idx)}')
print(f'Wrong            : {len(errors_idx)}')
print(f'Avg confidence correct : {test_probs[correct_idx].mean():.4f}')
print(f'Avg confidence wrong   : {test_probs[errors_idx].mean():.4f}')

In [ ]:

BEST_THRESHOLD = 0.35   

torch.save(best_model.state_dict(), 'artifacts/best_model.pt')

model_config = {
    'input_dim':     INPUT_DIM,
    'hidden_units':  [128, 64,32],
    'dropout_rate':  0.3,
    'use_batchnorm': True,
    'threshold':     BEST_THRESHOLD
}
with open('artifacts/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)


print('Files in artifacts:', os.listdir('artifacts'))

In [ ]:
with mlflow.start_run(run_name='best_model_final'):
    mlflow.log_params(model_config)
    mlflow.log_metrics({'test_acc': test_acc, 'test_auc': test_auc})
    mlflow.pytorch.log_model(best_model, 'best_model')
    mlflow.log_artifacts('artifacts', artifact_path='artifacts')

print('Best model logged to MLflow.')

In [ ]:
loaded_model = MLP()
loaded_model.load_state_dict(torch.load('artifacts/best_model.pt', map_location='cpu'))
loaded_model.eval()

with torch.no_grad():
    sample_probs = torch.sigmoid(loaded_model(X_test_t[:5])).numpy().flatten()

print('Sample probabilities:', sample_probs.round(4))
print('Model loaded and verified successfully.')

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.inspection import permutation_importance

# ── Wrap PyTorch model ──
class TorchModelWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model, device):
        self.model  = model
        self.device = device

    def fit(self, X, y):
        return self

    def predict(self, X):
        tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        self.model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(self.model(tensor)).cpu().numpy().flatten()
        return (probs > 0.5).astype(int)

    def score(self, X, y):
        preds = self.predict(X)
        return accuracy_score(y, preds)


# Feature names 
feature_names = [
    # Original 23
    'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
    'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
    'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
    # Engineered
    'TOTAL_DELAY', 'MAX_DELAY', 'CURRENTLY_DELAYED', 'CONSECUTIVE_DELAY',
    'AVG_BILL', 'UTILIZATION', 'BILL_TREND', 'MAX_BILL',
    'TOTAL_PAID', 'AVG_PAID', 'PAY_TREND', 'ZERO_PAY_COUNT',
    'BAL_AFTER_PAY1', 'BAL_AFTER_PAY2', 'BAL_AFTER_PAY3',
    'AVG_REMAINING_BAL', 'BAL_GROWING',
    'AGE_GROUP', 'HIGH_EDUCATION', 'LIMIT_PER_AGE'
]

# Verify count matches
print(f'Feature names : {len(feature_names)}')
print(f'X_test shape  : {X_test.shape[1]}')
assert len(feature_names) == X_test.shape[1], \
    f'Mismatch — {len(feature_names)} names vs {X_test.shape[1]} features'

# Run permutation importance 
wrapped_model = TorchModelWrapper(best_model, device)

result = permutation_importance(
    wrapped_model,
    X_test,
    test_labels,
    n_repeats=10,
    random_state=42,
    scoring='recall'
)


importance_df = pd.DataFrame({
    'feature':    feature_names,
    'importance': result.importances_mean,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('\n=== Feature Importance (by Recall impact) ===\n')
print(importance_df.to_string(index=False))


original_features = [
    'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
    'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
    'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
]

engineered_features = [
    'TOTAL_DELAY', 'MAX_DELAY', 'CURRENTLY_DELAYED', 'CONSECUTIVE_DELAY',
    'AVG_BILL', 'UTILIZATION', 'BILL_TREND', 'MAX_BILL',
    'TOTAL_PAID', 'AVG_PAID', 'PAY_TREND', 'ZERO_PAY_COUNT',
    'BAL_AFTER_PAY1', 'BAL_AFTER_PAY2', 'BAL_AFTER_PAY3',
    'AVG_REMAINING_BAL', 'BAL_GROWING',
    'AGE_GROUP', 'HIGH_EDUCATION', 'LIMIT_PER_AGE'
]


top5_original    = importance_df[importance_df['feature'].isin(original_features)].head(5).reset_index(drop=True)
top5_engineered  = importance_df[importance_df['feature'].isin(engineered_features)].head(5).reset_index(drop=True)


print('\nTop 5 Original Features \n')
for i, row in top5_original.iterrows():
    print(f'  {i+1}. {row["feature"]:15s}  importance: {row["importance"]:.4f}')


print('\nTop 5 Engineered Features \n')
for i, row in top5_engineered.iterrows():
    print(f'  {i+1}. {row["feature"]:20s}  importance: {row["importance"]:.4f}')


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

# Chart 1 — Top 20 overall
top20  = importance_df.head(20)
colors = ['tomato' if x > 0 else 'steelblue' for x in top20['importance']]
ax1.barh(top20['feature'][::-1], top20['importance'][::-1],
         color=colors[::-1], edgecolor='white')
ax1.set_title('Top 20 Overall Features')
ax1.set_xlabel('Drop in Recall when shuffled')
ax1.axvline(0, color='black', linewidth=0.8)


# Chart 2 — Top 5 original
ax2.barh(top5_original['feature'][::-1], top5_original['importance'][::-1],
         color='steelblue', edgecolor='white')
ax2.set_title('Top 5 Original Features')
ax2.set_xlabel('Drop in Recall when shuffled')
ax2.axvline(0, color='black', linewidth=0.8)

# Chart 3 — Top 5 engineered
ax3.barh(top5_engineered['feature'][::-1], top5_engineered['importance'][::-1],
         color='tomato', edgecolor='white')
ax3.set_title('Top 5 Engineered Features')
ax3.set_xlabel('Drop in Recall when shuffled')
ax3.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Feature Importance — Impact on Recall', fontsize=13)
plt.tight_layout()
plt.savefig('artifacts/feature_importance.png', dpi=100)
plt.show()

# For Streamlit 
print(f'TOP_FEATURES            = {top5_original["feature"].tolist()}')
print(f'TOP_ENGINEERED_FEATURES = {top5_engineered["feature"].tolist()}')

In [ ]:
import json

#  get predictions on test set 
_, test_acc, test_probs, test_labels = evaluate(best_model, test_loader, criterion)
test_preds = (test_probs > BEST_THRESHOLD).astype(int)
test_labels_int = test_labels.astype(int)

# inverse transform to get original scale values 
X_test_original = scaler.inverse_transform(X_test)

#  find indices for each quadrant 
tp_idx = np.where((test_preds == 1) & (test_labels_int == 1))[0]
tn_idx = np.where((test_preds == 0) & (test_labels_int == 0))[0]
fp_idx = np.where((test_preds == 1) & (test_labels_int == 0))[0]
fn_idx = np.where((test_preds == 0) & (test_labels_int == 1))[0]

# pick most confident example from each quadrant 
tp_best = tp_idx[np.argmax(test_probs[tp_idx])]   
tn_best = tn_idx[np.argmin(test_probs[tn_idx])]   
fp_best = fp_idx[np.argmax(test_probs[fp_idx])]   
fn_best = fn_idx[np.argmin(test_probs[fn_idx])]   

col_names = [
    'LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE',
    'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6',
    'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6',
    'PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6',
]

def extract_23(row_idx):
    row = X_test_original[row_idx]
    vals = {col_names[i]: row[i] for i in range(23)}
    # round sensibly
    for k in ['LIMIT_BAL','BILL_AMT1','BILL_AMT2','BILL_AMT3',
              'BILL_AMT4','BILL_AMT5','BILL_AMT6',
              'PAY_AMT1','PAY_AMT2','PAY_AMT3',
              'PAY_AMT4','PAY_AMT5','PAY_AMT6']:
        vals[k] = int(round(vals[k]))
    for k in ['SEX','EDUCATION','MARRIAGE','AGE',
              'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']:
        vals[k] = int(round(vals[k]))
    return vals

results = {
    "TP": {"idx": int(tp_best), "prob": float(test_probs[tp_best]), "values": extract_23(tp_best)},
    "TN": {"idx": int(tn_best), "prob": float(test_probs[tn_best]), "values": extract_23(tn_best)},
    "FP": {"idx": int(fp_best), "prob": float(test_probs[fp_best]), "values": extract_23(fp_best)},
    "FN": {"idx": int(fn_best), "prob": float(test_probs[fn_best]), "values": extract_23(fn_best)},
}

print(json.dumps(results, indent=2))